In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
import openpyxl

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


In [2]:


# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

XLSX_URL = (
    "https://www.data.gouv.fr/api/1/datasets/r/06d9816c-1b87-498d-985e-f312acee4f51"
)

path_xlsx_raw     = os.path.join(RAW_DIR, "2022_burvot_t2_france_entiere.xlsx")
path_rhone_bronze = os.path.join(BRONZE_DIR, "2022_bronze_burvot_t2_rhone_69.csv")



In [3]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_xlsx_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(XLSX_URL, timeout=180)
    resp.raise_for_status()
    with open(path_xlsx_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_xlsx_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


📥 Downloading source to RAW...
✅ Saved to RAW: ../../../data/raw/2022_burvot_t2_france_entiere.xlsx


In [4]:

# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_all = pd.read_excel(path_xlsx_raw, header=0, dtype=str, engine="openpyxl")

# Identify the department column
col_dept = next(c for c in df_all.columns if any(kw in c.lower() for kw in ["département", "dept"]))

# Filter for Rhône (69)
df_bronze = df_all[df_all[col_dept].astype(str).str.strip().str.lstrip("0") == "69"].copy()
print("____________________")
print(df_bronze.columns.to_list())

# # --- RECONSTRUCT CANDIDATE COLUMNS (Structural Ingestion) ---
# CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
# cols = list(df_bronze.columns)
# start_cand = next((i for i, c in enumerate(cols) if "panneau" in c.lower()), None)

# if start_cand is not None:
#     new_cols = cols[:start_cand]
#     # Calculate candidates based on remaining original columns
#     n_cand = (len(cols) - start_cand) // len(CAND_FIELDS)
#     for k in range(n_cand):
#         suffix = f".{k}" if k > 0 else ""
#         for field in CAND_FIELDS:
#             new_cols.append(f"{field}{suffix}")
#     df_bronze.columns = new_cols

# --- ADD BRONZE METADATA ---
# Essential for the Medallion pattern to track data lineage
df_bronze['extraction_source_url'] = XLSX_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']     = os.path.basename(path_xlsx_raw)

# ── 4. SAVE TO BRONZE ───────────────────────────────────────────────────────
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")
display(df_bronze.head())

print(f"✅ BRONZE dataset created: {path_rhone_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


🛠 Processing RAW -> BRONZE...
____________________
['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32']


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Etat saisie,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,extraction_source_url,ingestion_timestamp,source_file_name
26977,69,Rhône,001,Affoux,Complet,316,49,15.51,267,84.49,21,6.65,7.87,9,2.85,3.37,237,75,88.76,1,M,MACRON,Emmanuel,98,31.01,41.35,2,F,LE PEN,Marine,139,43.99,58.65,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26978,69,Rhône,002,Aigueperse,Complet,217,47,21.66,170,78.34,9,4.15,5.29,4,1.84,2.35,157,72.35,92.35,1,M,MACRON,Emmanuel,74,34.1,47.13,2,F,LE PEN,Marine,83,38.25,52.87,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26979,69,Rhône,003,Albigny-sur-Saône,Complet,1860,504,27.1,1356,72.9,81,4.35,5.97,18,0.97,1.33,1257,67.58,92.7,1,M,MACRON,Emmanuel,823,44.25,65.47,2,F,LE PEN,Marine,434,23.33,34.53,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26980,69,Rhône,004,Alix,Complet,470,95,20.21,375,79.79,35,7.45,9.33,4,0.85,1.07,336,71.49,89.6,1,M,MACRON,Emmanuel,194,41.28,57.74,2,F,LE PEN,Marine,142,30.21,42.26,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26981,69,Rhône,005,Ambérieux,Complet,407,62,15.23,345,84.77,10,2.46,2.9,3,0.74,0.87,332,81.57,96.23,1,M,MACRON,Emmanuel,151,37.1,45.48,2,F,LE PEN,Marine,181,44.47,54.52,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx


✅ BRONZE dataset created: ../../../data/bronze/2022_bronze_burvot_t2_rhone_69.csv
📊 Rows: 267 | Columns: 36


In [5]:
df_bronze.dtypes

pd.set_option('display.max_columns', None)

display(df_bronze.head())


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Etat saisie,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,extraction_source_url,ingestion_timestamp,source_file_name
26977,69,Rhône,001,Affoux,Complet,316,49,15.51,267,84.49,21,6.65,7.87,9,2.85,3.37,237,75,88.76,1,M,MACRON,Emmanuel,98,31.01,41.35,2,F,LE PEN,Marine,139,43.99,58.65,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26978,69,Rhône,002,Aigueperse,Complet,217,47,21.66,170,78.34,9,4.15,5.29,4,1.84,2.35,157,72.35,92.35,1,M,MACRON,Emmanuel,74,34.1,47.13,2,F,LE PEN,Marine,83,38.25,52.87,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26979,69,Rhône,003,Albigny-sur-Saône,Complet,1860,504,27.1,1356,72.9,81,4.35,5.97,18,0.97,1.33,1257,67.58,92.7,1,M,MACRON,Emmanuel,823,44.25,65.47,2,F,LE PEN,Marine,434,23.33,34.53,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26980,69,Rhône,004,Alix,Complet,470,95,20.21,375,79.79,35,7.45,9.33,4,0.85,1.07,336,71.49,89.6,1,M,MACRON,Emmanuel,194,41.28,57.74,2,F,LE PEN,Marine,142,30.21,42.26,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26981,69,Rhône,005,Ambérieux,Complet,407,62,15.23,345,84.77,10,2.46,2.9,3,0.74,0.87,332,81.57,96.23,1,M,MACRON,Emmanuel,151,37.1,45.48,2,F,LE PEN,Marine,181,44.47,54.52,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx


In [6]:
candidate_cols = [
    "N°Panneau.1",
    "Sexe.1",
    "Nom.1",
    "Prénom.1",
    "Voix.1",
    "% Voix/Ins.1",
    "% Voix/Exp.1"
]

for i, col in enumerate(df_bronze.columns):
    if "Unnamed:" in col:
        df_bronze = df_bronze.rename(columns={col: candidate_cols.pop(0)})

In [7]:
df_bronze.dtypes

pd.set_option('display.max_columns', None)

display(df_bronze.head())


,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Etat saisie,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,N°Panneau.1,Sexe.1,Nom.1,Prénom.1,Voix.1,% Voix/Ins.1,% Voix/Exp.1,extraction_source_url,ingestion_timestamp,source_file_name
26977,69,Rhône,001,Affoux,Complet,316,49,15.51,267,84.49,21,6.65,7.87,9,2.85,3.37,237,75,88.76,1,M,MACRON,Emmanuel,98,31.01,41.35,2,F,LE PEN,Marine,139,43.99,58.65,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26978,69,Rhône,002,Aigueperse,Complet,217,47,21.66,170,78.34,9,4.15,5.29,4,1.84,2.35,157,72.35,92.35,1,M,MACRON,Emmanuel,74,34.1,47.13,2,F,LE PEN,Marine,83,38.25,52.87,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26979,69,Rhône,003,Albigny-sur-Saône,Complet,1860,504,27.1,1356,72.9,81,4.35,5.97,18,0.97,1.33,1257,67.58,92.7,1,M,MACRON,Emmanuel,823,44.25,65.47,2,F,LE PEN,Marine,434,23.33,34.53,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26980,69,Rhône,004,Alix,Complet,470,95,20.21,375,79.79,35,7.45,9.33,4,0.85,1.07,336,71.49,89.6,1,M,MACRON,Emmanuel,194,41.28,57.74,2,F,LE PEN,Marine,142,30.21,42.26,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
26981,69,Rhône,005,Ambérieux,Complet,407,62,15.23,345,84.77,10,2.46,2.9,3,0.74,0.87,332,81.57,96.23,1,M,MACRON,Emmanuel,151,37.1,45.48,2,F,LE PEN,Marine,181,44.47,54.52,https://www.data.gouv.fr/api/1/datasets/r/06d9...,2026-03-18T14:29:31.512838,2022_burvot_t2_france_entiere.xlsx
